### Machine learning and materials physics, 05-13-2026

You will follow a complete machine learning pipeline, from data preparation to model evaluation, using a materials science dataset. 

**Instructions:**
- Answer the questions in the cells provided for this purpose, using code if necessary.
- Online resources will be helpful! See e.g. the documentation of scikit-learn
- Questions are in <span style="color: blue;">blue</span>.
- The bonus part at the end of the notebook is not necessarily more complicated than the rest.

**Subject:**
We will be using a library (which you likely haven't used until now) called `matminer`. This is a library that allows to grab and exploit materials science datasets. After preparing this data, we will implement several machine learning models using the features of the `scikit-learn` library. Throughout the exam, we will use `matplotlib` to present results in graphical form.

The dataset we will use was the subject of a scientific article, which you can find in PDF format on Colab (`elastic_tensor.pdf`), or [online](https://www.nature.com/articles/sdata20159) (open-access). It is a dataset covering the elastic properties of many inorganic crystalline compounds (metals, oxides, alloys, etc.). These properties were not measured experimentally but were calculated using quantum chemistry methods.

---

### **Part 1: Data Preparation and Exploration**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matminer.datasets import load_dataset
from matminer.featurizers.composition import ElementProperty
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

from sklearn.model_selection import KFold, cross_val_score

Here we load the "elastic_tensor_2015" dataset, which was previously downloaded, into a `pandas` DataFrame. We also remove some columns of data that we will not need.

In [ ]:
df = pd.read_pickle("elastic_tensor.pkl")

unwanted_columns = ["volume", "nsites", "compliance_tensor", "elastic_tensor", 
                    "elastic_tensor_original", "K_Voigt", "G_Voigt", "K_Reuss", "G_Reuss"]
df = df.drop(unwanted_columns, axis=1)

<span style="color: blue;">**Question 1**: Display the first 5 rows, then a statistical summary (`.describe()`) of the DataFrame.</span>

<span style="color: blue;">**Question 2**: How many data points are there? What is the dimensionality of this data?</span>

<span style="color: blue;">**Question 3**: Based on the article `elastic_tensor.pdf`, determine which column label corresponds to the shear modulus in our filtered DataFrame. What units are used?</span>

<span style="color: blue;">**Question 4**: Prepare four graphs below representing the distributions of the columns `elastic_anisotropy`, `G_VRH`, `K_VRH`, and `poisson_ratio` using histograms. For example, you could use the `plt.hist()` method from `matplotlib`. For the first quantity, use a logarithmic scale for the y-axis.</span>

The "formula" column contains the chemical formula of the materials. We cannot use it directly in a machine learning model for regression, for example (as it is a text string); with `matminer`, we will generate new descriptors from this formula. This is done here in four steps: 
1) Convert the text string into a "composition" in the matminer format
2) Retrieve descriptors specific to this elemental composition, as well as
3) Oxidation state descriptors
4) Descriptors of the crystal structure of the materials, obtained from the density

In [ ]:
from matminer.featurizers.conversions import StrToComposition
from matminer.featurizers.composition import ElementProperty
from matminer.featurizers.conversions import CompositionToOxidComposition
from matminer.featurizers.composition import OxidationStates
from matminer.featurizers.structure import DensityFeatures

df = StrToComposition().featurize_dataframe(df, "formula")

ep_feat = ElementProperty.from_preset(preset_name="magpie", impute_nan=True)
df = ep_feat.featurize_dataframe(df, col_id="composition")

df = CompositionToOxidComposition().featurize_dataframe(df, "composition")

os_feat = OxidationStates()
df = os_feat.featurize_dataframe(df, "composition_oxid")

df_feat = DensityFeatures()
df = df_feat.featurize_dataframe(df, "structure")

<span style="color: blue;">**Question 5** : What is the new dimensionality of the data?</span>

144 (obtained for instance via `df.describe()`). 

### **Part 2: A first linear model**

We will now try to predict the bulk modulus $K$, using the descriptors introduced previously. This modulus describes a material's resistance to isostatic compression: 

$$K = -V \frac{dP}{dV},$$ 

where $V$ is the initial volume of the material and $P$ is the pressure. 
Below, we prepare two variables, `y` and `X`, which correspond respectively to the bulk modulus and the set of descriptors with which we will attempt to predict it. `X` therefore forms a basis on which we will try to express `y`.

In [ ]:
y = df['K_VRH'].values
excluded = ["G_VRH", "K_VRH", "elastic_anisotropy", "formula", "material_id", 
            "poisson_ratio", "structure", "composition", "composition_oxid"]
X = df.drop(excluded, axis=1)

print("There are {} descriptors:\n\n{}".format(X.shape[1], X.columns.values))

<span style="color: blue;">**Question 6**: Prepare a linear regression model to estimate the bulk modulus from the descriptors listed previously. For example, you could use the `LinearRegression` class from `scikit-learn`. For now, do not split the data into training and test sets.</span>

<span style="color: blue;">**Question 7**: Report the value of the coefficient of determination ($R^2$), as well as the root-mean-square error (RMSE). These two quantities should be calculated on the entire dataset.</span>

We now wish to perform a $k$-fold cross-validation.

<span style="color: blue;">**Question 8**: Perform this cross-validation with $k=10$ on our linear model. You could, for example, use a class already implemented in `scikit-learn`. You will report two scores: the $R^2$ and the RMSE, obtained by averaging across the distribution of folds. Comment on their values compared to what you obtained earlier by training the model with all the data.</span>

<span style="color: blue;">**Question 9**: Now report the standard deviations of the $R^2$ and RMSE distributions across all folds.</span>

### **Part 3: A Random forest model**

We now wish to implement a more complex model: a [random forest](https://en.wikipedia.org/wiki/Random_forest), also known as a forest of decision trees. The goal is to determine if we can obtain a more performant or robust model than the previous linear regression.

![title](img/random_forest.png)

<span style="color: blue;">**Question 10**: Implement a random forest model (**regression!**) composed of 50 decision trees. At this stage, use all the data for training.</span>

<span style="color: blue;">**Question 11**: Report the $R^2$ and the RMSE for this model. Have we obtained a more accurate model than the previous linear regression?</span>

<span style="color: blue;">**Question 12**: Perform a $k$-fold cross-validation on this model, with $k=10$.</span>

<span style="color: blue;">**Question 13**: Report the means and standard deviations of the $R^2$ and RMSE across the fold distributions. By comparing these values to those obtained with the linear model, can we say that we have obtained a model that is more or less accurate, and more or less robust?</span>

<span style="color: blue;">**Question 14**: Now train a single random forest model, this time with a data split into a training set (80%) and a test set (20%). For example, you could use the `train_test_split()` method from `scikit-learn`. Once the model is trained, prepare a graph with the reference bulk modulus on the $x$-axis and the model's prediction on the $y$-axis. Present the training and validation data using different colors. Print also the training and test set error metrics. </span>

### **Part 4: Neural networks**

We now wish to implement a neural network regression model. We will use the Keras library (from TensorFlow), which implements relatively simple neural networks. In the following cell, there is code that implements a feed-forward neural network. 

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler

# 1. Data Scaling (Important for Neural Networks)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Build the Model Architecture
# We use a Sequential model with a few dense layers
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1) # Output layer for regression (no activation)
])

# 3. Compile the Model
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

# 4. Train the Model
# We'll set a small number of epochs for this exercise
history = model.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

# 5. Model Prediction
y_pred_nn = model.predict(X_test_scaled).flatten()

# 6. Evaluation
print(f"Neural Network - MAE: {mean_absolute_error(y_test, y_pred_nn):.4f}")
print(f"Neural Network - R2: {r2_score(y_test, y_pred_nn):.4f}")

<span style="color: blue;">**Question 15**: Now play around with the architecture, the optimizer, etc! </span>